# 8. Build and evaluate a tiny search system
## Goal
Represent text numerically, retrieve source passages, and separate evidence retrieval from answer generation.
Course Module 12. The six documents and evaluation queries below are invented. There is no LLM, API request, or generated answer in this notebook.
## Setup
Uses scikit-learn and NumPy. The collection is a fixed search corpus; fitting its TF-IDF vocabulary is indexing available documents, not fitting on held-back question labels.
Reference: [Hugging Face course](https://huggingface.co/learn/llm-course/chapter1/1) for the subsequent language-model concepts.
## Steps
### 1. Index the documents

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = {
    "refunds": "Refunds are returned to the original payment method within seven business days.",
    "delivery": "Standard delivery takes three days. Express delivery arrives the next day.",
    "hours": "Customer support hours are Monday to Friday from nine to five.",
    "password": "Reset your password using the account settings page and your email address.",
    "cancellation": "Cancel an order before dispatch by contacting customer support.",
    "privacy": "Export or delete your account data from the privacy settings page.",
}
ids = list(documents)
vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents.values())
print("Document-term matrix shape:", document_vectors.shape)

Document-term matrix shape: (6, 35)


### 2. Return evidence with a bounded fallback
A zero-overlap fallback is simple and incomplete: nonzero similarity does not prove that a question is answerable.

In [2]:
def retrieve(question, k=2):
    if not isinstance(question, str) or not question.strip():
        return []
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, document_vectors).ravel()
    ranked = np.argsort(-scores)
    return [{"id": ids[i], "score": float(scores[i]), "text": documents[ids[i]]}
            for i in ranked[:k] if scores[i] > 0]

for hit in retrieve("How long do refunds take?"):
    print(hit)
assert retrieve("How long do refunds take?")[0]["id"] == "refunds"
assert retrieve("") == []

{'id': 'refunds', 'score': 0.3610220414609507, 'text': 'Refunds are returned to the original payment method within seven business days.'}


### 3. Evaluate a fixed small question set
For each answerable query, one relevant document is labeled. Hit@k asks whether it appears among the top k. With one relevant document per query here, this is also recall@k. The tiny hand-authored set is a smoke test, not a reliable estimate of production quality.

In [3]:
evaluation = [
    ("When will refunds arrive?", "refunds"),
    ("How fast is express delivery?", "delivery"),
    ("What are customer support hours?", "hours"),
    ("How do I reset my password?", "password"),
    ("Can I cancel before dispatch?", "cancellation"),
    ("How can I delete my account data?", "privacy"),
]
top1, top2 = [], []
for question, relevant_id in evaluation:
    found = [hit["id"] for hit in retrieve(question)]
    top1.append(relevant_id in found[:1])
    top2.append(relevant_id in found[:2])
    print(question, "→", found, "expected:", relevant_id)
print(f"Hit@1: {np.mean(top1):.1%}; Hit@2: {np.mean(top2):.1%}")
assert all(isinstance(value, bool) for value in top1)

When will refunds arrive? → ['refunds'] expected: refunds
How fast is express delivery? → ['delivery'] expected: delivery
What are customer support hours? → ['hours', 'cancellation'] expected: hours
How do I reset my password? → ['password'] expected: password
Can I cancel before dispatch? → ['cancellation'] expected: cancellation
How can I delete my account data? → ['privacy', 'password'] expected: privacy
Hit@1: 100.0%; Hit@2: 100.0%


### 4. Find a failure that a good-looking score hides
Both questions below are outside the corpus. Shared vocabulary can still retrieve a passage.

In [4]:
unknown_questions = ["What is the capital of Peru?", "Do refunds include a cryptocurrency bonus?"]
for question in unknown_questions:
    hits = retrieve(question)
    print(question, "→", [hit["id"] for hit in hits] or "No matching terms")
print("A retrieved refund policy does not answer whether cryptocurrency bonuses exist.")
print("Singular/plural mismatch:", retrieve("How long does a refund take?"))
print("Without stemming, refund and refunds are different terms, so evidence can be missed.")

What is the capital of Peru? → No matching terms
Do refunds include a cryptocurrency bonus? → ['refunds']
A retrieved refund policy does not answer whether cryptocurrency bonuses exist.
Singular/plural mismatch: []
Without stemming, refund and refunds are different terms, so evidence can be missed.


## Checks
Explain why the system has not answered the question merely by retrieving a passage. Explain the difference between lexical TF-IDF and a learned embedding, and between indexing documents and training a generator. The deliberately irrelevant 'refunds' question exposes the weakness of the fallback even if the answerable-query score is high.
## Next Steps
Complete exercise 38. Write five new questions before changing the retriever. Add ambiguous and unsupported cases. The 40-hour bridge then requires a real pretrained model and a document assistant with separately measured retrieval, correctness, evidence support, latency, and optional provider cost.